In [1]:
import pandas as pd
import numpy as np

## Textual Summarization:
Text summarization is mainly done using Pre-Trained large scale Transformer model OR LLM like GPT-3.

We will here look at different transformer model, and design class to use it.

In [2]:
df = pd.read_parquet("../Dataset/Clean/Dataset_with_clusters.parquet")
print(df.columns)
df.sample(3)

Index(['Content', 'Summary', 'Dataset', 'char_count', 'sentence_count',
       'word_count', 'unique_word_count', 'lexical_diversity', 'hapax_ratio',
       'stopword_count', 'stopword_ratio', 'noun_count', 'verb_count',
       'adj_count', 'adv_count', 'pronoun_count', 'person_count', 'org_count',
       'gpe_count', 'event_count', 'unique_entity_count',
       'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog',
       'char_count_summary', 'sentence_count_summary', 'word_count_summary',
       'unique_word_count_summary', 'lexical_diversity_summary',
       'hapax_ratio_summary', 'stopword_count_summary',
       'stopword_ratio_summary', 'noun_count_summary', 'verb_count_summary',
       'adj_count_summary', 'adv_count_summary', 'pronoun_count_summary',
       'person_count_summary', 'org_count_summary', 'gpe_count_summary',
       'event_count_summary', 'unique_entity_count_summary',
       'flesch_reading_ease_summary', 'flesch_kincaid_grade_summary',
       'gunning_fog_

,Content,Summary,Dataset,char_count,sentence_count,word_count,unique_word_count,lexical_diversity,hapax_ratio,stopword_count,...,flesch_kincaid_grade_summary,gunning_fog_summary,Log_content_char_count,Log_summary_char_count,word_ratio,char_ratio,sent_ratio,Embedding,Cluster,Topic
4594,During a session of the devolution of the rail...,With the meeting of London Assembly's Transpor...,XSum,1549,12,254,147,0.578740,0.413386,111,...,16.053636,21.527273,7.346010,5.049856,0.086614,0.100065,0.083333,"[0.01780235394835472, -0.08153774589300156, 0....",4,0
6640,All of the best jobs require bucket loads of e...,A Berlin sex business is looking for a man or ...,CNN/Daily Mail,1968,16,310,168,0.541935,0.370968,132,...,7.398378,8.176577,7.585281,5.429346,0.119355,0.115346,0.187500,"[-0.05398571118712425, -0.02672225423157215, -...",9,5
2695,"Bergsson, 51, is standing for election as pres...",Former England internationals Gary Neville and...,XSum,1756,14,285,174,0.610526,0.442105,122,...,16.041053,13.915789,7.471363,4.997212,0.063158,0.083713,0.071429,"[-0.0033753456082195044, -0.04062987118959427,...",1,1


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "sshleifer/distilbart-cnn-12-6" # or "facebook/bart-large-cnn" or "Falconsai/text_summarization"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

d:\Krishan\Project dataset\News categorization\.NewsEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- We switch to AutoTokenizer because large transformers 

In [4]:
import torch
from pydantic import BaseModel, field_validator, ValidationError

class ChunkRequest(BaseModel):
    text: str
    
    @field_validator("text")
    @classmethod
    def validate_text(cls, v):
        if not v.strip() or not isinstance(v, str):
            raise ValidationError("Text must be a non-empty string")
        return v.strip()
    
class SummarizeRequest(BaseModel):
    text: str
    
    @field_validator("text")
    @classmethod
    def validate_text(cls, v):
        if not v.strip() or not isinstance(v, str):
            raise ValidationError("Text must be a non-empty string")
        return v.strip()

In [49]:
## First we need a Text Chunker to bypass token limit
from typing import List
import html, re, unicodedata
from bs4 import BeautifulSoup
import spacy

class TextChunker:
    def __init__(self, tokenizer=None, max_tokens:int=512, clean_whitespace:bool=True):
        """
        Tokenizer: Hugging face tokenizer - best suitable for transformers
        max_tokens: max tokens per chunk (model dependent)
        """
        self.tokenizer = tokenizer if tokenizer else AutoTokenizer.from_pretrained(model_name)
        self.max_tokens = max_tokens
        self.clean_whitespace = clean_whitespace
        self.nlp = spacy.load("en_core_web_sm")
        
    def _clean_text(self, text:str) -> str:
        """
        Perform safe, canonical text cleaning for NLP tasks.
        Preserves linguistic structure.
        """
        if not self.clean_whitespace:
            return text
        
        if pd.isna(text) or not isinstance(text, str):
            return ''
        
        # 1. Fix broken encoding and HTML entities
        s = html.unescape(text)    
        # 2. normalize unicode (NFKC helps)
        s = unicodedata.normalize('NFKC', s)
        # 3. Remove HTML tags (robust)
        s = BeautifulSoup(s, 'lxml').get_text(separator=" ")
        # 4. remove ZERO WIDTH and BOM chars
        s = re.sub(r'[\u200B-\u200D\uFEFF]', '', s)
        # 5. Normalize whitespace (spaces, tabs)
        s = re.sub(r"[ \t]+", " ", s)
        # 6. Remove repeated newlines
        s = re.sub(r"\n\s*\n+", "\n", s)
        # 7. Strip leading and trailing whitespace
        s = s.strip()
        
        return s
    
    def _split_sentence(self, text:str) -> List[str]:
        """
        Basic sentence splitting using regex
        You can replace with spaCy or nltk if needed
        """
        # sentences = re.split(r'(?<=[,!?])\s+', text)
        # return [s.strip() for s in sentences if s.strip()]
        ## OR else
        doc = self.nlp(text)
        return [str(sent) for sent in doc.sents]
    
    def chunk(self, text:str) -> List[str]:
        req = ChunkRequest(text=text)
        text = self._clean_text(req.text)
        sentences = self._split_sentence(text)
        
        chunks = []
        curr_chunk = []
        current_len = 0
        
        for sent in sentences:
            sent_len = len(self.tokenizer(str(sent), add_special_tokens=False)["input_ids"])
            
            # If sentence itself exceeds max_token, give it its own chunk
            if sent_len > self.max_tokens:
                if curr_chunk:
                    chunks.append(" ".join(curr_chunk))
                    # chunks.append(curr_chunk)
                    curr_chunk, current_len = [], 0
                
                chunks.append(sent)
                continue
            
            # If adding sentence exceeds limit -> flush chunk
            if current_len+sent_len > self.max_tokens:
                chunks.append(" ".join(curr_chunk)) # add previous
                # chunks.append(curr_chunk) # add previous
                
                # reset new sent into next chunk
                curr_chunk = [sent]
                current_len = sent_len
            else: # adding sent keep chunk within bound -> add furthur
                curr_chunk.append(sent)
                current_len += sent_len
                
        if curr_chunk: # add last chunk
            chunks.append(" ".join(curr_chunk))
            # chunks.append(curr_chunk)
        
        return chunks


In [73]:
from typing import Tuple, Optional

class NewsSummarizer:
    def __init__(self, model_name=None, tokenizer=None, summarizer=None):
        self.tokenizer = tokenizer if tokenizer else AutoTokenizer.from_pretrained(model_name)
        self.model = summarizer if summarizer else AutoModelForSeq2SeqLM.from_pretrained(model_name)
        
        if (not self.tokenizer) or (not self.model):
            raise Exception("Model and Tokenizer not loaded")
        
        self.Chunker = TextChunker(tokenizer=self.tokenizer, max_tokens=256)
    
    def _get_summary_length(self, text:str, num_chunks:int, compression:float) -> Tuple[int, int]:
        word_count = len(self.tokenizer.encode(text))
        
        if compression is None: compression = 0.5
        
        target = max(50, int(word_count*compression))
        min_len = max(10, int(target*0.5))
        
        return min_len, target
    
    def summarize(self, text:str, compression:Optional[float]=0.5) -> str:
        req = SummarizeRequest(text=text)
        text = req.text
        
        chunks = self.Chunker.chunk(text)
        
        summarizes = []
        
        for ck in chunks:
            min_len, max_len = self._get_summary_length(text=ck, num_chunks=len(chunks), compression=compression)
            
            inputs = self.tokenizer(ck, return_tensors="pt", truncation=True)
            
            with torch.no_grad():
                output_ids = self.model.generate(
                    inputs["input_ids"],
                    min_length=min_len, max_length=max_len,
                    num_beams=4, no_repeat_ngram_size=3,
                    # repetition_penalty=1.15, length_penalty=2.0, 
                    early_stopping=True, do_sample=False # deterministic
                )
            summary = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
            
            summarizes.append(summary)
            
        return "\n".join(summarizes)
    

In [28]:
query = df.sample(1)["Content"].values[0]
print("Query: ", query)

Query:  Tax credit changes could hit three million families, which are likely to lose an average of £1,000, it said.
Even taking into account higher wages, people receiving tax credits would be "significantly worse off", said Paul Johnson, director of the IFS.
The chancellor said most workers would be better off under Budget changes.
The Budget also included the introduction of a National Living Wage, which is to be introduced from next year. George Osborne said on Wednesday that it would rise to £9 an hour by 2020.
A spokeswoman for the Prime Minister, David Cameron, dismissed criticism of the Budget as "regressive", saying that it was designed to boost take-home pay while reducing benefits and welfare.
She added: "It keeps us on the path to stronger economic security, with lower spending and an economy which lives within its means."
The biggest impact on families will come from the freeze in working-age benefits and the changes to tax credits, said Mr Johnson of the IFS.
"It will red

In [74]:
Summarizer_model = NewsSummarizer(tokenizer=tokenizer, summarizer=model)
print("Article: \n", query)
print("- x -"*10, "\n")

summ = Summarizer_model.summarize(query)
print("Summary: \n", summ)

Article: 
 Tax credit changes could hit three million families, which are likely to lose an average of £1,000, it said.
Even taking into account higher wages, people receiving tax credits would be "significantly worse off", said Paul Johnson, director of the IFS.
The chancellor said most workers would be better off under Budget changes.
The Budget also included the introduction of a National Living Wage, which is to be introduced from next year. George Osborne said on Wednesday that it would rise to £9 an hour by 2020.
A spokeswoman for the Prime Minister, David Cameron, dismissed criticism of the Budget as "regressive", saying that it was designed to boost take-home pay while reducing benefits and welfare.
She added: "It keeps us on the path to stronger economic security, with lower spending and an economy which lives within its means."
The biggest impact on families will come from the freeze in working-age benefits and the changes to tax credits, said Mr Johnson of the IFS.
"It will 

> We can batch chunks at inference as well for faster processing

In [ ]:
def summarize(self, text: str, compression: float = 0.25) -> str:

    chunks = self.Chunker.chunk(text)

    min_len, max_len = self._get_summary_length(text, chunks, compression)

    # 🔥 BATCH TOKENIZATION
    inputs = self.tokenizer(
        chunks,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    )

    inputs = {k: v.to(self.device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = self.model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_len,
            min_length=min_len,
            num_beams=2,                     # faster
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    summaries = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    return " ".join(summaries)
